# Lesson 17: A PWR Unit Cell and the Four Factors

In this notebook, we'll

1. construct one reflected PWR unit cell;
2. compare the fuel, cladding, and coolant spectra;
3. form reaction-rate-preserving effective cross sections;
4. evaluate $\epsilon$, $p$, $f$, and $\eta_T$ from the handout definitions; and
5. compute the $k$ eigenvalue.


## Model and group definitions

The baseline follows the source tutorial: fuel radius 0.45 cm, cladding
outer radius 0.46 cm, pitch 1.20 cm, 4 wt% enrichment, 800 ppm soluble
boron, 1200 K fuel, and 600 K coolant at 15.5 MPa. The one-centimeter-high
square prism has reflective boundaries, so it represents an infinite
lattice with no leakage.

The handout's incident-energy groups are

$$
T: E<1\ \mathrm{eV},\qquad
I: 1\ \mathrm{eV}\le E<0.1\ \mathrm{MeV},\qquad
F: E\ge 0.1\ \mathrm{MeV}.
$$

`flux`, `absorption`, and `nu-fission` are tallied with both a `CellFilter`
and an `EnergyFilter`. OpenMC's reaction-rate scores already include the
spatial and energy integrals; division by volume or flux is a later analysis
step. Light water uses `c_H_in_H2O` thermal scattering data.


In [ ]:
import os
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openmc
import pandas as pd

cross_sections = os.environ.get("OPENMC_CROSS_SECTIONS")
if not cross_sections or not Path(cross_sections).is_file():
    raise RuntimeError(
        "Set OPENMC_CROSS_SECTIONS to an installed OpenMC "
        "continuous-energy cross_sections.xml file."
    )
openmc.config["cross_sections"] = cross_sections

REGION_NAMES = ("fuel", "cladding", "coolant")
GROUP_NAMES = ("T", "I", "F")
GROUP_EDGES = np.array([1.0e-5, 1.0, 1.0e5, 2.0e7])  # eV
SPECTRUM_EDGES = np.geomspace(1.0e-5, 2.0e7, 161)    # eV
SPECTRUM_CENTERS = np.sqrt(SPECTRUM_EDGES[:-1] * SPECTRUM_EDGES[1:])
DELTA_U = np.log(SPECTRUM_EDGES[1:] / SPECTRUM_EDGES[:-1])

PURPLE = "#512888"
ORANGE = "#CA7C1B"
TEAL = "#008080"


## 1. Build the heterogeneous cell

The builder returns both an `openmc.Model` and the bookkeeping needed to
normalize tallies. Volumes are for a unit axial height. The square and axial
boundaries are reflective; the fuel and cladding surfaces are transmitting.


In [ ]:
def build_pwr_model(
    *,
    fuel_radius=0.45,
    clad_radius=0.46,
    pitch=1.20,
    enrichment=4.0,
    boron_ppm=800.0,
    fuel_temperature=1200.0,
    moderator_temperature=600.0,
    moderator_pressure=15.5,
    particles=2000,
    batches=60,
    inactive=10,
    seed=17001,
):
    '''Build a reflected PWR cell with spectrum and three-group tallies.'''
    if pitch <= 2.0 * clad_radius:
        raise ValueError("pitch must exceed the cladding diameter")

    fuel = openmc.Material(name="UO2 fuel")
    fuel.add_element("U", 1.0, enrichment=enrichment)
    fuel.add_element("O", 2.0)
    fuel.set_density("g/cm3", 10.5)
    fuel.temperature = float(fuel_temperature)

    cladding = openmc.Material(name="zirconium cladding")
    cladding.add_element("Zr", 1.0)
    cladding.set_density("g/cm3", 6.6)
    cladding.temperature = 0.5 * (fuel_temperature + moderator_temperature)

    coolant = openmc.model.borated_water(
        boron_ppm=boron_ppm,
        temperature=moderator_temperature,
        pressure=moderator_pressure,
        temp_unit="K",
        press_unit="MPa",
        name="borated light water",
    )
    # borated_water attaches c_H_in_H2O thermal scattering.
    coolant.temperature = float(moderator_temperature)

    fuel_surface = openmc.ZCylinder(r=fuel_radius)
    clad_surface = openmc.ZCylinder(r=clad_radius)
    x_min = openmc.XPlane(x0=-pitch / 2.0, boundary_type="reflective")
    x_max = openmc.XPlane(x0=+pitch / 2.0, boundary_type="reflective")
    y_min = openmc.YPlane(y0=-pitch / 2.0, boundary_type="reflective")
    y_max = openmc.YPlane(y0=+pitch / 2.0, boundary_type="reflective")
    z_min = openmc.ZPlane(z0=-0.5, boundary_type="reflective")
    z_max = openmc.ZPlane(z0=+0.5, boundary_type="reflective")
    prism = +x_min & -x_max & +y_min & -y_max & +z_min & -z_max

    cells = (
        openmc.Cell(name="fuel", fill=fuel, region=-fuel_surface & prism),
        openmc.Cell(
            name="cladding",
            fill=cladding,
            region=+fuel_surface & -clad_surface & prism,
        ),
        openmc.Cell(name="coolant", fill=coolant, region=+clad_surface & prism),
    )
    volumes = np.array(
        [
            np.pi * fuel_radius**2,
            np.pi * (clad_radius**2 - fuel_radius**2),
            pitch**2 - np.pi * clad_radius**2,
        ]
    )
    for cell, volume in zip(cells, volumes, strict=True):
        cell.volume = float(volume)

    settings = openmc.Settings(run_mode="eigenvalue")
    settings.batches = int(batches)
    settings.inactive = int(inactive)
    settings.particles = int(particles)
    settings.seed = int(seed)
    settings.source = openmc.IndependentSource(
        space=openmc.stats.Point((0.0, 0.0, 0.0)),
        angle=openmc.stats.Isotropic(),
        energy=openmc.stats.Watt(),
    )
    settings.temperature = {"method": "interpolation"}

    cell_filter = openmc.CellFilter(cells)
    spectrum = openmc.Tally(name="regional spectrum")
    spectrum.filters = [cell_filter, openmc.EnergyFilter(SPECTRUM_EDGES)]
    spectrum.scores = ["flux"]

    rates = openmc.Tally(name="three-group rates")
    rates.filters = [cell_filter, openmc.EnergyFilter(GROUP_EDGES)]
    rates.scores = ["flux", "absorption", "nu-fission"]

    model = openmc.Model(
        geometry=openmc.Geometry(cells),
        materials=openmc.Materials([fuel, cladding, coolant]),
        settings=settings,
        tallies=openmc.Tallies([spectrum, rates]),
    )
    case = {
        "cells": cells,
        "materials": (fuel, cladding, coolant),
        "volumes": volumes,
        "fuel_radius": fuel_radius,
        "clad_radius": clad_radius,
        "pitch": pitch,
        "water_density": coolant.get_mass_density(),
    }
    return model, case


In [ ]:
model, case = build_pwr_model()
pd.Series(
    {
        "fuel radius [cm]": case["fuel_radius"],
        "cladding radius [cm]": case["clad_radius"],
        "pitch [cm]": case["pitch"],
        "coolant density [g/cm3]": case["water_density"],
        "fuel volume fraction": case["volumes"][0] / case["pitch"]**2,
        "coolant volume fraction": case["volumes"][2] / case["pitch"]**2,
    },
    name="baseline PWR cell",
)


## 2. Run the eigenvalue calculation

A temporary directory keeps `materials.xml`, `geometry.xml`, statepoints,
and other OpenMC products out of the course repository. The directory
remains alive until this notebook's kernel is stopped.


In [ ]:
run_directory = tempfile.TemporaryDirectory(prefix="ne630_lesson17_")
statepoint_path = model.run(cwd=run_directory.name, output=True)
statepoint_path


## 3. Compare the regional spectra

A flux tally is a track-length integral. For region $r$ and lethargy bin
$g$, divide by $V^r\Delta u_g$ to obtain a regional volume-average spectrum
per unit lethargy. The eigenvalue source normalization is arbitrary, but all
three curves share it and can therefore be compared directly.


In [ ]:
with openmc.StatePoint(statepoint_path) as statepoint:
    keff = statepoint.keff

    spectrum_tally = statepoint.get_tally(name="regional spectrum")
    spectrum_flux = spectrum_tally.get_values(
        scores=["flux"], value="mean"
    ).reshape(len(REGION_NAMES), len(SPECTRUM_CENTERS)).copy()

    rate_tally = statepoint.get_tally(name="three-group rates")
    group_flux = rate_tally.get_values(
        scores=["flux"], value="mean"
    ).reshape(len(REGION_NAMES), len(GROUP_NAMES)).copy()
    absorption = rate_tally.get_values(
        scores=["absorption"], value="mean"
    ).reshape(len(REGION_NAMES), len(GROUP_NAMES)).copy()
    production = rate_tally.get_values(
        scores=["nu-fission"], value="mean"
    ).reshape(len(REGION_NAMES), len(GROUP_NAMES)).copy()

flux_per_lethargy = spectrum_flux / case["volumes"][:, None] / DELTA_U

fig, ax = plt.subplots(figsize=(8, 4))
for region, values, color in zip(
    REGION_NAMES, flux_per_lethargy, (PURPLE, ORANGE, TEAL), strict=True
):
    ax.stairs(values, SPECTRUM_EDGES, label=region, color=color, linewidth=1.4)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(SPECTRUM_EDGES[[0, -1]])
ax.set_xlabel("Energy [eV]")
ax.set_ylabel(r"regional $\overline{\phi}(u)$ [arbitrary units]")
ax.grid(which="both", alpha=0.2)
ax.legend()
ax.set_title("PWR unit-cell spectra per unit lethargy");


The resonance depressions are strongest in the fuel, while the thermal
flux level differs among all three regions. That is why atom-weighting a
mixed material cannot, by itself, preserve heterogeneous reaction rates.

**Check before continuing:** at which energies is the assumption of one
spatially uniform flux least credible?


## 4. Start from rates; derive effective cross sections second

For group $g$ and region $r$,

$$
\Phi_g^r=\int_{V^r}\!\int_g\phi\,dE\,dV,\qquad
A_g^r=\int_{V^r}\!\int_g\Sigma_a\phi\,dE\,dV,\qquad
B_g^r=\int_{V^r}\!\int_g\nu\Sigma_f\phi\,dE\,dV.
$$

Thus $A_g^r/\Phi_g^r$ and $B_g^r/\Phi_g^r$ are spectrum-weighted
macroscopic cross sections. The rates themselves are the quantities needed
for the balance and factor definitions.


In [ ]:
rate_index = pd.MultiIndex.from_product(
    [REGION_NAMES, GROUP_NAMES], names=["region", "group"]
)
rate_table = pd.DataFrame(
    {
        "Phi": group_flux.ravel(),
        "A": absorption.ravel(),
        "B": production.ravel(),
    },
    index=rate_index,
)
rate_table


In [ ]:
sigma_a_effective = np.divide(
    absorption,
    group_flux,
    out=np.full_like(absorption, np.nan),
    where=group_flux > 0.0,
)

# Reaction-rate homogenization preserves sum_r A_g^r by construction.
sigma_a_homogenized = absorption.sum(axis=0) / group_flux.sum(axis=0)

# This comparison deliberately ignores the unequal regional group fluxes.
sigma_a_volume_weighted = np.average(
    sigma_a_effective, axis=0, weights=case["volumes"]
)

pd.DataFrame(
    {
        "rate-preserving Sigma_a [1/cm]": sigma_a_homogenized,
        "simple volume average [1/cm]": sigma_a_volume_weighted,
        "relative difference":
            sigma_a_volume_weighted / sigma_a_homogenized - 1.0,
    },
    index=pd.Index(GROUP_NAMES, name="group"),
)


## 5. Evaluate the four factors

The Lesson 17 retained-term model uses

$$
\epsilon=\frac{B_T+B_F}{B_T},\qquad
p=\frac{A_T}{A_T+A_I^f},\qquad
f=\frac{A_T^f}{A_T},\qquad
\eta_T=\frac{B_T}{A_T^f}.
$$

Here $A_T$ includes fuel, cladding, coolant, and dissolved boron.
Intermediate fission, fast absorption, and nonfuel nonthermal absorption
are deliberately omitted from this reduced product. Those omissions must
remain visible when the result is compared with the reference $k$ eigenvalue.


In [ ]:
T, I, F = range(3)
FUEL, CLADDING, COOLANT = range(3)

A_T = absorption[:, T].sum()
A_I_fuel = absorption[FUEL, I]
B_T = production[FUEL, T]
B_F = production[FUEL, F]

epsilon = (B_T + B_F) / B_T
p = A_T / (A_T + A_I_fuel)
f = absorption[FUEL, T] / A_T
eta_T = B_T / absorption[FUEL, T]

k_four_factor = epsilon * p * f * eta_T
k_retained_balance = (B_T + B_F) / (A_T + A_I_fuel)
B_over_A = production.sum() / absorption.sum()

np.testing.assert_allclose(k_four_factor, k_retained_balance, rtol=1.0e-12)

pd.Series(
    {
        "epsilon": epsilon,
        "p": p,
        "f": f,
        "eta_T": eta_T,
        "k, four-factor": k_four_factor,
        "k, retained-rate balance": k_retained_balance,
        "k, full-energy balance (B/A)": B_over_A,
        "OpenMC keff": float(keff.n),
        "OpenMC keff standard deviation": float(keff.s),
    },
)


In [ ]:
pd.Series(
    {
        "omitted intermediate fission production B_I": production[FUEL, I],
        "omitted fast absorption A_F (all regions)": absorption[:, F].sum(),
        "omitted nonfuel intermediate absorption A_I": absorption[1:, I].sum(),
    },
    name="rates outside the retained-term model",
)


In this continuous-energy model, intermediate fission production
is large enough that the approximate 4-factor formula is not expected to reproduce
`keff`. The full-energy $B/A$ comparison tests the no-leakage neutron balance
without hiding the omitted terms.


## 6. The thermal disadvantage

The volume-average thermal-flux ratio is

$$
\zeta=\frac{\overline{\phi}_{mT}}{\overline{\phi}_{fT}}
=\frac{\Phi_T^m/V_m}{\Phi_T^f/V_f}.
$$

The handout's two-region expression includes fuel and moderator. Our
detailed direct-rate $f$ also includes cladding absorption, so the two
values need not be identical.


In [ ]:
phi_bar_T = group_flux[:, T] / case["volumes"]
zeta = phi_bar_T[COOLANT] / phi_bar_T[FUEL]

f_two_region = 1.0 / (
    1.0
    + zeta
    * case["volumes"][COOLANT]
    * sigma_a_effective[COOLANT, T]
    / (case["volumes"][FUEL] * sigma_a_effective[FUEL, T])
)

pd.Series(
    {
        "zeta = phi_mT / phi_fT": zeta,
        "two-region f (fuel + coolant)": f_two_region,
        "direct-rate f (fuel + clad + coolant)": f,
        "cladding share of thermal absorption": absorption[CLADDING, T] / A_T,
    },
    name="thermal disadvantage",
)


## Interpretation

1. Which omitted rate explains most of the difference between the
   four-factor product and full-energy $B/A$?
2. Why does direct reaction-rate homogenization differ most from simple
   volume weighting in the resonance and thermal groups?
3. Which factor should respond most directly to soluble boron? Which should
   respond most directly to additional moderation?
4. Compare this thermal spectrum with the Lesson 16 SFR spectrum. Which
   assumptions that were tolerable for the fast cell fail here?

Lesson 18 turns these questions into controlled pitch, boron, fuel-temperature,
and moderator-density studies.
